# 激活函数手撕实现

| 函数 | 定义 | 导数 | 主要用途 / 注意 |
|------|------|------|------------------|
| ReLU | $\max(0,x)$ | $x>0?1:0$ | 简单、稀疏；负区梯度恒为 0（死亡 ReLU） |
| Sigmoid | $\frac1{1+e^{-x}}$ | $s(1-s)$ | 输出 ∈(0,1)；饱和区梯度消失 |
| Tanh | $\frac{e^x-e^{-x}}{e^x+e^{-x}}$ | $1-t^2$ | 零中心；仍饱和 |
| SiLU/Swish | $x\cdot\sigma(x)$ | $\sigma(x)+x\sigma(x)(1-\sigma(x))$ | LLaMA 默认，平滑、非单调 |
| GELU | $x\cdot\Phi(x)$ | — | BERT/GPT 用，近似 $0.5x(1+\tanh[\frac{2}{\pi}^{1/2}(x+0.044715x^3)])$ |

**为什么大模型用 SiLU/GELU 而非 ReLU**：平滑可导、负区保留小梯度，缓解死亡单元；非单调带来更好表达。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ReLU(nn.Module):
    def forward(self, x):
        return torch.maximum(x, torch.zeros_like(x))   # 不用 max(x,0.0) 以支持广播/设备

class Sigmoid(nn.Module):
    def forward(self, x):
        return 1.0 / (1.0 + torch.exp(-x))

class Tanh(nn.Module):
    def forward(self, x):
        return torch.tanh(x)

class SiLU(nn.Module):
    def forward(self, x):
        return x * torch.sigmoid(x)

class GELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (1.0 + torch.erf(x / torch.sqrt(torch.tensor(2.0))))

In [ ]:
# 与 PyTorch 内置对拍
x = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])
for name, fn, ref in [
    ('ReLU',  ReLU(),  F.relu),
    ('Sigmoid', Sigmoid(), torch.sigmoid),
    ('Tanh',  Tanh(),   torch.tanh),
    ('SiLU',  SiLU(),   F.silu),
    ('GELU',  GELU(),   F.gelu),
]:
    out = fn(x)
    print(f'{name:8s}: {out.tolist()}  match={torch.allclose(out, ref(x))}')

In [ ]:
# 导数验证（autograd vs 解析）
x = torch.randn(5, requires_grad=True)
s = torch.sigmoid(x)
s.sum().backward()
print('sigmoid grad autograd:', x.grad.tolist())
print('sigmoid grad analytic:', (torch.sigmoid(x.detach()) * (1 - torch.sigmoid(x.detach()))).tolist())

## 小结 / 易错点
- `torch.max(x, 0.0)` 在新版本会广播报错，用 `torch.maximum(x, zeros_like(x))` 更稳。
- Sigmoid 导数用前向值 $s(1-s)$，不要重算 exp。
- SiLU 非单调，负区不为 0，这是它优于 ReLU 的关键。
- GELU 精确式用 `erf`，近似式用 tanh 展开（推理常融合）。